## News Category Classification Model
### Full Fine-tuning DistilBert model

In [1]:
# generate hugginface token or login to hf account
import huggingface_hub
huggingface_hub.login()

In [2]:
# import the necessary libraries and modules
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate
    import gradio as gr

import torch
import transformers
import random
import numpy as np
import pandas as pd

# see version of the libraries
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"evaluate version: {evaluate.__version__}")
print(f"accelerate version: {accelerate.__version__}")
print(f"gradio version: {gr.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
transformers version: 5.16.1
datasets version: 4.0.0
evaluate version: 0.4.6
accelerate version: 1.14.0
gradio version: 6.26.0


### Prepare Dataset

In [3]:
from datasets import load_dataset

# load the dataset
dataset = load_dataset(path="AiresPucrs/News-Category-Dataset")
dataset

README.md:   0%|          | 0.00/847 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/209527 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 209527
    })
})

In [4]:
print(dataset.column_names)
print(dataset['train'].features)
print(dataset['train'][0])

{'train': ['text', 'labels']}
{'text': Value('string'), 'labels': Value('string')}
{'text': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.', 'labels': 'U.S. NEWS'}


In [5]:
# unique labels in the dataset
labels = dataset['train'].unique("labels")
print(f"Unique labels: {labels}")
print(f"Number of unique labels: {len(labels)}")

Unique labels: ['U.S. NEWS', 'COMEDY', 'PARENTING', 'WORLD NEWS', 'CULTURE & ARTS', 'TECH', 'SPORTS', 'ENTERTAINMENT', 'POLITICS', 'WEIRD NEWS', 'ENVIRONMENT', 'EDUCATION', 'CRIME', 'SCIENCE', 'WELLNESS', 'BUSINESS', 'STYLE & BEAUTY', 'FOOD & DRINK', 'MEDIA', 'QUEER VOICES', 'HOME & LIVING', 'WOMEN', 'BLACK VOICES', 'TRAVEL', 'MONEY', 'RELIGION', 'LATINO VOICES', 'IMPACT', 'WEDDINGS', 'COLLEGE', 'PARENTS', 'ARTS & CULTURE', 'STYLE', 'GREEN', 'TASTE', 'HEALTHY LIVING', 'THE WORLDPOST', 'GOOD NEWS', 'WORLDPOST', 'FIFTY', 'ARTS', 'DIVORCE']
Number of unique labels: 42


In [6]:
# turn the dataset into a pandas dataframe and check some samples
news_df = pd.DataFrame(dataset['train'])
news_df.sample(5)

,text,labels
95811,Former Rep. Barney Frank: Justice Scalia Is A ...,POLITICS
97221,Barber Who Slashed Customer's Throat Heads To ...,CRIME
109642,Mike Huckabee Discusses Those Beyonce Comments,ENTERTAINMENT
179104,Holiday Stress: 8 Seasonal Ways To Chill Out S...,WELLNESS
85792,These Carts Are Letting Kids With Disabilities...,GOOD NEWS


In [7]:
# create mapping of labels to numeric values
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

Label to ID mapping: {'U.S. NEWS': 0, 'COMEDY': 1, 'PARENTING': 2, 'WORLD NEWS': 3, 'CULTURE & ARTS': 4, 'TECH': 5, 'SPORTS': 6, 'ENTERTAINMENT': 7, 'POLITICS': 8, 'WEIRD NEWS': 9, 'ENVIRONMENT': 10, 'EDUCATION': 11, 'CRIME': 12, 'SCIENCE': 13, 'WELLNESS': 14, 'BUSINESS': 15, 'STYLE & BEAUTY': 16, 'FOOD & DRINK': 17, 'MEDIA': 18, 'QUEER VOICES': 19, 'HOME & LIVING': 20, 'WOMEN': 21, 'BLACK VOICES': 22, 'TRAVEL': 23, 'MONEY': 24, 'RELIGION': 25, 'LATINO VOICES': 26, 'IMPACT': 27, 'WEDDINGS': 28, 'COLLEGE': 29, 'PARENTS': 30, 'ARTS & CULTURE': 31, 'STYLE': 32, 'GREEN': 33, 'TASTE': 34, 'HEALTHY LIVING': 35, 'THE WORLDPOST': 36, 'GOOD NEWS': 37, 'WORLDPOST': 38, 'FIFTY': 39, 'ARTS': 40, 'DIVORCE': 41}
ID to Label mapping: {0: 'U.S. NEWS', 1: 'COMEDY', 2: 'PARENTING', 3: 'WORLD NEWS', 4: 'CULTURE & ARTS', 5: 'TECH', 6: 'SPORTS', 7: 'ENTERTAINMENT', 8: 'POLITICS', 9: 'WEIRD NEWS', 10: 'ENVIRONMENT', 11: 'EDUCATION', 12: 'CRIME', 13: 'SCIENCE', 14: 'WELLNESS', 15: 'BUSINESS', 16: 'STYLE & BE

In [8]:
# Map the labels in the dataset to their corresponding numeric values
def map_labels(example):
    example['labels'] = label2id[example['labels']]
    return example

label_mapped_dataset = dataset.map(map_labels)
# check some sample data after mapping
print(label_mapped_dataset['train'][5:10])

Map:   0%|          | 0/209527 [00:00<?, ? examples/s]

{'text': ['Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her family reported her missing, authorities said.', 'Reporter Gets Adorable Surprise From Her Boyfriend While Live On TV "Who\'s that behind you?" an anchor for New York’s PIX11 asked journalist Michelle Ross as she finished up an interview.', 'Puerto Ricans Desperate For Water After Hurricane Fiona’s Rampage More than half a million people remained without water service three days after the storm lashed the U.S. territory.', 'How A New Documentary Captures The Complexity Of Being A Child Of Immigrants In "Mija," director Isabel Castro combined music documentaries with the style of "Euphoria" and "Clueless" to tell a more nuanced immigration story.', "Biden At UN To Call Russian War An Affront To Body's Charter White House officials say the crux of the president's visit to the U.N. this year wi

In [9]:

label_mapped_dataset['train'].shuffle()[:5]


{'text': ["Rumer Willis Shows Off Cleavage In Barely-There Top (PHOTOS) See more wardrobe malfunctions! Rumer Willis doesn't exactly frequent best-dressed lists, but her latest look has us wondering",
  "Training For A Marathon Doesn't Just Make You Awesome -- It's Good For Your Heart The researchers recruited these men to participate in an 18-week training regimen, which included endurance training, group",
  "Miley Cyrus Brings Back The 'Hannah Montana' Look For 'Ew!' It's the best of both worlds.",
  'Actually, Academics And Athletics Do Mix Pretty Well Starting UCLA QB Josh Rosen gave an interview to Bleacher Report where he claimed that football and school don’t go together.',
  'Smart Travel Advice: The More You Travel, the Bigger the World Becomes "The world is essentially a friendly place and the people you encounter -- whatever their backgrounds and beliefs -- are more similar to you than different from you," he adds. "Ask for locals\' advice and help on the road, and your jou

In [10]:
from datasets import DatasetDict

# split the dataset into train, validation and test sets
train_test_val_dataset = label_mapped_dataset['train'].train_test_split(test_size=0.2, seed=42)
# This results in 10% validation and 10% test relative to the original data
test_valid = train_test_val_dataset["test"].train_test_split(test_size=0.5, seed=42)

# Pack everything into a unified DatasetDict
final_dataset = DatasetDict({
    "train": train_test_val_dataset["train"],
    "validation": test_valid["train"],  # The 'train' part of the second split
    "test": test_valid["test"]          # The 'test' part of the second split
})

print(final_dataset)
final_dataset['test'].shuffle()[:5]


DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
})


{'text': ["Gisele Bundchen, Tom Brady Copy Kimye, Step Out In His & Hers Leather Outfits (PHOTOS) Meanwhile, check out the true pioneers of matchy couple outfits: Gisele, who's the new face of Chanel, looks predictably",
  "7 Things You Didn't Know You Could Do on a Cruise Gone are the days when the height of cruise entertainment was limited to poolside lounging and fancy evening soirees.",
  'Guest Etiquette: How NOT To Overstay Your Welcome (VIDEO) Click through our slideshow of beach houses we hope we get an invite to. ** Watch the video clip above for some house guest',
  'Megyn Kelly On Donald Trump: \'I Certainly Will Not Apologize For Doing Good Journalism\' "Mr. Trump did interviews over the week that attacked me personally. I\'ve decided not to respond."',
  'Reflections On Dr. Samuel DuBois Cook: A Great Teacher And Role Model When Dr. Samuel DuBois Cook passed away May 29 our nation and world lost a very creative and distinguished political scientist'],
 'labels': [16, 23, 2

### Prepare Tokenizer

In [11]:
from transformers import AutoTokenizer

# load the tokenizer for the model we want to use
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased", use_fast=True)

tokenizer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [12]:
# test the tokenizer on a sample text
sample_text = "This is a sample text for tokenization."
tokenized_output = tokenizer(sample_text)
tokenized_output

{'input_ids': [101, 2023, 2003, 1037, 7099, 3793, 2005, 19204, 3989, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [13]:
# check the maximum sequence length of the tokenizer
max_seq_length = tokenizer.model_max_length
vocab_size = tokenizer.vocab_size
print(f"Maximum sequence length of the tokenizer: {max_seq_length}, vocabulary size: {vocab_size}")

Maximum sequence length of the tokenizer: 512, vocabulary size: 30522


In [14]:
# define a tokenization function to apply to the dataset texts
def tokenize_text(example):
    return tokenizer(example['text'], padding=True, truncation=True)

# test the function on a sample text
sample_example= {'text': "The official Google Colab extension for VS Code does not natively support the Secrets (Key icon) user-data feature. Because the extension runs the notebook inside the VS Code Jupyter interface rather than the standard web UI, the google.colab.userdata module will fail to fetch keys stored in your browser-based Colab secrets panel", 'labels': 5}

tokenized_text_sample = tokenize_text(sample_example)
tokenized_text_sample

{'input_ids': [101, 1996, 2880, 8224, 15270, 2497, 5331, 2005, 5443, 3642, 2515, 2025, 3128, 2135, 2490, 1996, 7800, 1006, 3145, 12696, 1007, 5310, 1011, 2951, 3444, 1012, 2138, 1996, 5331, 3216, 1996, 14960, 2503, 1996, 5443, 3642, 18414, 7685, 3334, 8278, 2738, 2084, 1996, 3115, 4773, 21318, 1010, 1996, 8224, 1012, 15270, 2497, 1012, 5310, 2850, 2696, 11336, 2097, 8246, 2000, 18584, 6309, 8250, 1999, 2115, 16602, 1011, 2241, 15270, 2497, 7800, 5997, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [15]:
final_dataset
# Map the tokenization function to the entire dataset
tokenized_dataset = final_dataset.map(function=tokenize_text, batched=True, batch_size=1000)
tokenized_dataset

Map:   0%|          | 0/167621 [00:00<?, ? examples/s]

Map:   0%|          | 0/20953 [00:00<?, ? examples/s]

Map:   0%|          | 0/20953 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
})

In [16]:
# see two samples from the tokenized dataset with their all keys
tokenized_dataset['train'].shuffle()[:2]

{'text': ["E.U. Austerity, You Must Be Kidding The leading political lights in Europe -- Messrs. Hollande, Valls and Macron in France and Mr. Renzi in Italy -- are raising a big stink about fiscal austerity. They don't like it. And now Greece has jumped on the anti-austerity bandwagon.",
  '10 Best Houseplants To De-Stress Your Home And Purify The Air Did you know bringing the outdoors in can help relieve tension?'],
 'labels': [38, 20],
 'input_ids': [[101,
   1041,
   1012,
   1057,
   1012,
   17151,
   3334,
   3012,
   1010,
   2017,
   2442,
   2022,
   12489,
   1996,
   2877,
   2576,
   4597,
   1999,
   2885,
   1011,
   1011,
   6752,
   2869,
   1012,
   7935,
   2063,
   1010,
   11748,
   4877,
   1998,
   26632,
   2078,
   1999,
   2605,
   1998,
   2720,
   1012,
   14916,
   5831,
   1999,
   3304,
   1011,
   1011,
   2024,
   6274,
   1037,
   2502,
   27136,
   2055,
   10807,
   17151,
   3334,
   3012,
   1012,
   2027,
   2123,
   1005,
   1056,
   2066,
   2009

### Evaluate Functions for the Model

In [17]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Computes the accuracy of a model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  # Get highest prediction probability of each prediction if predictions are probabilities
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

In [18]:
# Create example list of predictions and labels for testing the evaluate function
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when one prediction is wrong: {'accuracy': 0.9}


### Model Training

In [19]:
# define model and load the pretained model for sequence classification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path= "distilbert/distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [21]:
# count the parameters in the model
def count_params(model):
    """
    Count the parameters of a PyTorch model.
    """
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_parameters = sum(p.numel() for p in model.parameters())

    return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

# Count the parameters of the model
count_params(model)

{'trainable_parameters': 66985770, 'total_parameters': 66985770}

In [22]:
# Create model output directory
from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "distilbert-base-uncased-News-Category-Classifier"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir


# define the saved model path (huggingface model hub path)
# model_save_name = "distilbert-base-uncased-News-Category-Classifier"
# model_save_path = f"{huggingface_hub.whoami()['name']}/{model_save_name}"
# model_save_path

PosixPath('models/distilbert-base-uncased-News-Category-Classifier')

In [26]:
# define training arguments for the model training
import torch
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

# Detect whether a GPU is actually available. Training on CPU-only vs. a GPU
# needs very different settings (batch size, mixed precision, epoch count),
# so we branch on this rather than guessing.
has_gpu = torch.cuda.is_available()
print(f"[INFO] CUDA available: {has_gpu}")

training_args = TrainingArguments(
    output_dir=model_save_dir,

    # --- learning rate ---
    # 1e-4 is too high for full fine-tuning of a pretrained transformer and risks
    # the loss diverging or the model forgetting its pretrained weights.
    # 2e-5 is the standard starting point for BERT-family fine-tuning.
    learning_rate=2e-5,
    # warmup_ratio=0.1,
    weight_decay=0.01,

    # --- batch size / memory ---
    # Small per-device batch keeps memory usage low on CPU or limited-VRAM GPUs.
    # gradient_accumulation_steps simulates a larger, more stable effective batch
    # (16 * 2 = 32) without needing enough memory to hold a batch of 32 at once.
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,   # eval has no gradients/optimizer state, so it's cheaper
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,     # trades some speed for a large reduction in memory use

    # --- mixed precision ---
    # fp16 only helps (and is only reliably supported) on CUDA GPUs; leave it off
    # on CPU/MPS where it does nothing or isn't supported.
    fp16=has_gpu,

    # --- epochs ---
    # 10 epochs multiplies an already scarce compute budget by 10x and invites
    # overfitting. 3 epochs is a reasonable starting point for fine-tuning on
    # 150k+ examples -- check the eval accuracy curve and extend only if it's
    # still clearly improving.
    num_train_epochs=3,

    # --- evaluation / checkpointing ---
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,              # keep disk usage down (was 3)
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # --- logging ---
    # Log by steps rather than only at epoch end -- on slow hardware a single
    # epoch can take a long time, and step-level logs confirm training is
    # actually progressing instead of leaving you guessing.
    logging_strategy="steps",
    logging_steps=100,

    # --- misc ---
    dataloader_num_workers=2,        # a couple of background workers speeds up data
                                      # loading without starving CPU-only training of cores
    seed=42,
    report_to="none",                # optional: log to Weights & Biases/similar (off for now)
    # push_to_hub=True,              # optional: automatically upload the model to the Hub
    # hub_token="your_token_here",   # optional: HF token to push (defaults to huggingface-cli login)
    hub_private_repo=False,          # optional: make the uploaded model private
)

effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f"[INFO] Effective train batch size: {effective_batch_size}")

[INFO] Saving model checkpoints to: models/distilbert-base-uncased-News-Category-Classifier
[INFO] CUDA available: True
[INFO] Effective train batch size: 64


In [27]:
# define the Trainer for model training and evaluation
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    compute_metrics = compute_accuracy
)

trainer

In [28]:
# train the model
train_results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.356578,1.124169,0.676037
2,1.912239,1.045600,0.692932
3,1.720688,1.031381,0.695604


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
# Inspect training metrics
for key, value in train_results.metrics.items():
    print(f"{key}: {value}")

train_runtime: 4263.9277
train_samples_per_second: 117.934
train_steps_per_second: 1.843
total_flos: 3.645339623249583e+16
train_loss: 2.209987756860165
epoch: 3.0


In [31]:
# Save model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

[INFO] Saving model to models/distilbert-base-uncased-News-Category-Classifier


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [32]:
# Save our model to the Hugging Face Hub
# This will be public, since we set hub_private_repo=False in our TrainingArguments
model_upload_url = trainer.push_to_hub(
    commit_message="Uploading news category text classifier model",
    # token="YOUR_HF_TOKEN_HERE" # This will default to the token you have saved in your Hugging Face config
)
print(f"[INFO] Model successfully uploaded to Hugging Face Hub with at URL: {model_upload_url}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO] Model successfully uploaded to Hugging Face Hub with at URL: https://huggingface.co/mahdi2020/distilbert-base-uncased-News-Category-Classifier/commit/862806d9e97e424f244215b059c0eb1ff6a46903


In [33]:
# Perform predictions on the test set
predictions_all = trainer.predict(tokenized_dataset["test"])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f"[INFO] Prediction metrics on the test data:")
prediction_metrics

[INFO] Prediction metrics on the test data:


{'test_loss': 1.0144916772842407,
 'test_accuracy': 0.7012360998425047,
 'test_runtime': 32.4626,
 'test_samples_per_second': 645.45,
 'test_steps_per_second': 20.177}

In [38]:
import torch
from sklearn.metrics import accuracy_score

# 1. Get prediction probabilities (this is optional, could get the same results with step 2 onwards)
pred_probs = torch.softmax(torch.tensor(prediction_values), dim=1)

# 2. Get the predicted labels
pred_labels = torch.argmax(pred_probs, dim=1)

# 3. Get the true labels
true_labels = final_dataset["test"]["labels"]

# 4. Compare predicted labels to true labels to get the test accuracy
test_accuracy = accuracy_score(y_true=true_labels,
                               y_pred=pred_labels)

print(f"[INFO] Test accuracy: {test_accuracy*100}%")

[INFO] Test accuracy: 70.12360998425046%


In [39]:
# Make a DataFrame of test predictions
test_predictions_df = pd.DataFrame({
    "text": tokenized_dataset["test"]["text"],
    "true_label": true_labels,
    "pred_label": pred_labels,
    "pred_prob": torch.max(pred_probs, dim=1).values
})

test_predictions_df.head()

,text,true_label,pred_label,pred_prob
0,Bathroom Pods Inspired By...Airstream Trailers...,20,20,0.985476
1,"The Authentic Yogini ""I always need the lesson...",14,14,0.875120
2,How I Got Healthy: Part One: Panic Attacks and...,35,14,0.613191
3,10 Vegetarian Dinners Even Meat-Eaters Will Lo...,34,34,0.703287
4,Kim Davis Loses Latest Gay Marriage Appeal Dav...,8,8,0.870879


In [40]:
# Show 10 examples with low prediction probability
test_predictions_df.sort_values("pred_prob", ascending=True).head(10)

,text,true_label,pred_label,pred_prob
7862,I Am A Princess Pan,21,35,0.122876
15762,Why I Want To Remain Twitter Illiterate I must...,39,39,0.126088
20752,Nature Is a Gift She noticed I could not take ...,39,14,0.127139
3781,A Dr. Seuss Kind of Story About Love and Its G...,2,7,0.127161
15595,Flying Squirrel Nickname: Offensive or Progres...,22,37,0.127948
15097,'Hunger Games' Tributes: Get To Know Katniss' ...,7,16,0.132394
2216,Instant Justice In A Traffic Jam Is The Most S...,9,8,0.133768
3415,Nazi-Looted Libraries: Items Must Be Returned ...,31,24,0.133928
17582,Letting My Southern Roots Grow When a writer g...,14,39,0.134242
13440,A Generation Without Friends A friend is no lo...,2,2,0.138031


### Model Inference

In [1]:
# get the model path from huggingface hub
hf_model_path = "mahdi2020/distilbert-base-uncased-News-Category-Classifier"

In [2]:
import torch
# set best device for the model inference
def set_device():
    """
    Set device to CUDA if available, else MPS (Mac), else CPU.

    This defaults to using the best available device (usually).
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    return device

DEVICE = set_device()
print(f"[INFO] Using device: {DEVICE}")

[INFO] Using device: cuda


In [3]:
# setup the prediction ppeline
import torch
from transformers import pipeline

# Set the batch size for predictions
BATCH_SIZE = 32

# Create an instance of transformers.pipeline
news_category_classifier = pipeline(task="text-classification",
                                    model=hf_model_path,
                                    tokenizer=hf_model_path,
                                    device=DEVICE, # set the target device
                                    top_k=5, # only return the top 5 predicted value
                                    batch_size=BATCH_SIZE) # perform predictions on up to BATCH_SIZE number of samples at a time


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

In [4]:
# make predictions on article headlines
news_category_classifier("Two 3.71km service bridges of Dhaka-Ashulia Elevated Expressway opened")

[[{'label': 'WORLDPOST', 'score': 0.7179966568946838},
  {'label': 'TRAVEL', 'score': 0.11149885505437851},
  {'label': 'IMPACT', 'score': 0.01775955781340599},
  {'label': 'ARTS', 'score': 0.017628313973546028},
  {'label': 'RELIGION', 'score': 0.017268676310777664}]]

In [5]:
news_category_classifier("Google launches AI-first design tool Google Pics")

[[{'label': 'TECH', 'score': 0.8500754237174988},
  {'label': 'BUSINESS', 'score': 0.08791186660528183},
  {'label': 'WORLDPOST', 'score': 0.007651933468878269},
  {'label': 'IMPACT', 'score': 0.005362889263778925},
  {'label': 'MEDIA', 'score': 0.005285363178700209}]]

In [6]:
news_category_classifier("Donald trump wins the election this year")

[[{'label': 'POLITICS', 'score': 0.9260405898094177},
  {'label': 'COMEDY', 'score': 0.031484950333833694},
  {'label': 'ENTERTAINMENT', 'score': 0.02014164812862873},
  {'label': 'MEDIA', 'score': 0.0031487010419368744},
  {'label': 'BUSINESS', 'score': 0.0025425043422728777}]]

In [12]:
# batch prediction for 10 news headlines
batch_news_category_classifier = pipeline(
    task="text-classification",
    model=hf_model_path,
    tokenizer=hf_model_path,
    device=DEVICE,
    top_k=1,
    batch_size=10
)

# sample news
all_news = [
    "11 Killed in US Strikes Across Iran Following Sudden Escalation in Hostilities",
    "Earth Set to Permanently Overshoot Paris Agreement’s 1.5-Degree Warming Threshold, UN Warns",
    "Global Oil Prices Jump 1% Amid Surging Fears of Middle East Supply Disruptions",
    "Only Those with Deep Networks Will Survive': Tech Sector Reels Under Aggressive AI Expansion",
    "NASA Clean Air Study: Expert Ranks the Top 15 Low-Maintenance Houseplants for Beginners",
    "Cricketer Aaryavir Sehwag Smashes Boundaries to Replicate Iconic Senior Sehwag Style",
    "Elderly Delhi Couple Swindled Out of Rs 30 Lakh in Multi-Day 'Digital Arrest' Scam",
    "Iran Outlines ""Unforgettable Lessons"" for Washington as Ballistic Missiles Target Regional US Bases",
    "Nepal Demands 'Climate Compensation' From Global Superpowers After Catastrophic Glacial Floods",
    "Trump Pressures Strained Refinery Executives to Artificially Lower Pump Prices for Consumers"
]

# prediction
batch_news_category_classifier(all_news)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[{'label': 'WORLDPOST', 'score': 0.8140015006065369}],
 [{'label': 'GREEN', 'score': 0.6356074810028076}],
 [{'label': 'BUSINESS', 'score': 0.5168614387512207}],
 [{'label': 'BUSINESS', 'score': 0.5585752725601196}],
 [{'label': 'ENVIRONMENT', 'score': 0.42830753326416016}],
 [{'label': 'SPORTS', 'score': 0.5706007480621338}],
 [{'label': 'WORLDPOST', 'score': 0.5566640496253967}],
 [{'label': 'WORLDPOST', 'score': 0.796410322189331}],
 [{'label': 'WORLDPOST', 'score': 0.40721654891967773}],
 [{'label': 'POLITICS', 'score': 0.8752222657203674}]]

In [14]:
# Making Prediction with Pytorch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# define the model path

hf_model_path = "mahdi2020/distilbert-base-uncased-News-Category-Classifier"

# Create an example to predict on
sample_text_news = "English Premier Leauge: Spurs 0-2 Newcastle on the opeing fixture 2. Antony Elanga scored the first one and Wissa the next"

# Prepare the tokenizer and tokenize the inputs
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=hf_model_path)
inputs = tokenizer(sample_text_news,
                    return_tensors="pt") # return the output as PyTorch tensors
# load the model
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=hf_model_path)

print(f"Inputs - {inputs}")
print(f"Model - {model}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Inputs - {'input_ids': tensor([[  101,  2394,  4239, 12203, 22890,  1024, 18205,  1014,  1011,  1016,
          8142,  2006,  1996,  6728, 12377,  2290, 15083,  1016,  1012, 16262,
          3449, 18222,  3195,  1996,  2034,  2028,  1998, 15536, 11488,  1996,
          2279,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}
Model - DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
   

In [16]:
# Make a prediction/inference
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted class and prediction probability
output_logits = outputs.logits
predicted_class_id = torch.argmax(output_logits, dim=1).item()
predicted_class_label = model.config.id2label[predicted_class_id]
predicted_probability = torch.softmax(output_logits, dim=1).max().item()

# Print outputs
print(f"Text: {sample_text_news}")
print(f"Predicted class: {predicted_class_label} (prob: {predicted_probability * 100:.2f}%)")

Text: English Premier Leauge: Spurs 0-2 Newcastle on the opeing fixture 2. Antony Elanga scored the first one and Wissa the next
Predicted class: SPORTS (prob: 96.95%)
